# PANTANAL-1 M03 — Baseline Inference Notebook

This is an **M03 baseline notebook**. It can produce either:

- **REAL_SAMPLE_ZERO_BASELINE** — zero probabilities using real Kaggle `sample_submission.csv` schema, writing `/kaggle/working/submission.csv` when that file is found on Kaggle.
- **SYNTHETIC_FALLBACK_ONLY** — M01/M02-style synthetic smoke when no real sample file exists.

**Non-claims:**

- Does not prove model inference or competitive quality (zero baseline only).
- Does not prove leaderboard submission, score, or active competition eligibility.
- Does not prove CPU 90-minute scoring runtime unless commit/submit-mode evidence is recorded.
- Does not prove Kaggle commit/submit-mode execution unless explicitly run and documented.
- Synthetic fallback does not prove real `sample_submission.csv` compatibility.

BirdCLEF+ 2026 final deadline has passed; treat this notebook as archival/reusable baseline work unless eligibility is directly evidenced.

In [ ]:
import os
import platform
import sys
import time
from pathlib import Path

_m03_t0 = time.perf_counter()

print("=== PANTANAL-1 M03 Baseline Inference Debug ===")
print("Python:", sys.version)
print("Platform:", platform.platform())
print("cwd:", Path.cwd())
print("sys.path first entries:", sys.path[:10])
print("KAGGLE_KERNEL_RUN_TYPE:", os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
print("KAGGLE_URL_BASE:", os.environ.get("KAGGLE_URL_BASE"))

for path in [Path("/kaggle"), Path("/kaggle/input"), Path("/kaggle/working"), Path("tmp")]:
    print(f"{path}: exists={path.exists()}")
    if path.exists():
        try:
            print(f"{path} children:", [p.name for p in list(path.iterdir())[:20]])
        except Exception as exc:
            print(f"{path} listing failed:", repr(exc))

In [ ]:
from pathlib import Path

try:
    from pantanal_1.kaggle_paths import (
        all_sample_submission_candidates,
        find_sample_submission_path,
        is_kaggle_environment,
    )
    from pantanal_1.sample_baseline import (
        build_zero_rows_from_sample,
        class_columns_from_fieldnames,
        read_sample_submission_csv,
        validate_real_sample_zero_rows,
        write_submission_csv_with_fieldnames,
    )
    from pantanal_1.submission_contract import (
        build_zero_submission_rows,
        validate_submission_rows,
        write_submission_csv,
    )
    from pantanal_1.synthetic_schema import SYNTHETIC_CLASS_LABELS, SYNTHETIC_SOUNDSCAPE_STEMS

    SOURCE = "pantanal_1 package import"
except ModuleNotFoundError as exc:
    print("Package import failed:", repr(exc))
    print("Falling back to inline M03 baseline implementation.")
    import csv

    SOURCE = "inline fallback"

    EXPLICIT_CANDIDATES = (
        Path("/kaggle/input/competitions/birdclef-2026/sample_submission.csv"),
        Path("/kaggle/input/birdclef-2026/sample_submission.csv"),
    )

    def is_kaggle_environment():
        return bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle").exists()

    def discover_sample_submission_paths(root=Path("/kaggle/input")):
        if not root.exists():
            return ()
        found = []
        for path in sorted(root.rglob("sample_submission.csv")):
            if path.is_file():
                found.append(path)
        return tuple(found)

    def all_sample_submission_candidates():
        explicit = list(EXPLICIT_CANDIDATES)
        discovered = list(discover_sample_submission_paths())
        seen = set()
        ordered = []
        for p in explicit + discovered:
            if p not in seen:
                seen.add(p)
                ordered.append(p)
        return tuple(ordered)

    def find_sample_submission_path():
        for path in all_sample_submission_candidates():
            if path.is_file():
                return path
        return None

    def read_sample_submission_csv(sample_path):
        with sample_path.open(newline="", encoding="utf-8") as handle:
            reader = csv.DictReader(handle)
            fieldnames = list(reader.fieldnames)
            rows = [dict(row) for row in reader]
        return fieldnames, rows

    def class_columns_from_fieldnames(fieldnames):
        if "row_id" not in fieldnames:
            raise ValueError("missing row_id")
        cols = [n for n in fieldnames if n != "row_id"]
        if not cols:
            raise ValueError("no class columns")
        return cols

    def build_zero_rows_from_sample(fieldnames, sample_rows):
        class_cols = class_columns_from_fieldnames(fieldnames)
        out = []
        for row in sample_rows:
            z = {"row_id": row["row_id"]}
            z.update({c: 0.0 for c in class_cols})
            out.append(z)
        return out

    def validate_real_sample_zero_rows(rows, fieldnames):
        class_cols = class_columns_from_fieldnames(fieldnames)
        for idx, row in enumerate(rows):
            if list(row.keys()) != list(fieldnames):
                raise ValueError(f"row {idx} column order mismatch")
            for c in class_cols:
                if float(row[c]) != 0.0:
                    raise ValueError(f"row {idx} {c} must be 0.0")

    def write_submission_csv_with_fieldnames(rows, output_path, fieldnames):
        validate_real_sample_zero_rows(rows, fieldnames)
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(fieldnames))
            writer.writeheader()
            writer.writerows(rows)
        return output_path

    SYNTHETIC_CLASS_LABELS = tuple(f"synthetic_class_{idx:03d}" for idx in range(234))
    SYNTHETIC_SOUNDSCAPE_STEMS = (
        "BC2026_Test_0001_S05_20250227_010002",
        "BC2026_Test_0002_S05_20250227_010003",
    )

    def build_segment_end_times(duration_seconds=60, window_seconds=5):
        return list(range(window_seconds, duration_seconds + 1, window_seconds))

    def build_row_id(stem, end_time):
        return f"{stem}_{end_time}"

    def build_zero_submission_rows(stems, class_labels):
        rows = []
        for stem in stems:
            for end_time in build_segment_end_times():
                row = {"row_id": build_row_id(stem, end_time)}
                row.update({label: 0.0 for label in class_labels})
                rows.append(row)
        return rows

    def validate_submission_rows(rows, class_labels):
        if len(class_labels) != 234:
            raise ValueError("expected 234 synthetic class labels")

    def write_submission_csv(rows, output_path, class_labels):
        output_path = Path(output_path)
        if output_path.name not in ("m03_synthetic_baseline.csv",):
            if str(output_path) != "/kaggle/working/submission.csv":
                raise ValueError("unexpected output path for synthetic write")
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fieldnames = ["row_id", *class_labels]
        with output_path.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        return output_path


print("Import/bootstrap complete. SOURCE =", SOURCE)
print("is_kaggle_environment:", is_kaggle_environment())

candidates = all_sample_submission_candidates()
print("sample_submission.csv candidates (count):", len(candidates))
for cand in candidates:
    print("  candidate:", cand, "exists=", cand.is_file())

selected_sample = find_sample_submission_path()
print("selected sample submission path:", selected_sample)

In [ ]:
if selected_sample is not None:
    BASELINE_MODE = "REAL_SAMPLE_ZERO_BASELINE"
    fieldnames, sample_rows = read_sample_submission_csv(selected_sample)
    class_cols = class_columns_from_fieldnames(fieldnames)
    rows = build_zero_rows_from_sample(fieldnames, sample_rows)
    output_path = Path("/kaggle/working/submission.csv")
    print("mode:", BASELINE_MODE)
    print("sample path:", selected_sample)
    print("class column count:", len(class_cols))
    print("row count from sample:", len(sample_rows))
else:
    BASELINE_MODE = "SYNTHETIC_FALLBACK_ONLY"
    class_labels = list(SYNTHETIC_CLASS_LABELS)
    stems = list(SYNTHETIC_SOUNDSCAPE_STEMS)
    rows = build_zero_submission_rows(stems, class_labels)
    output_path = Path("tmp/submissions/m03_synthetic_baseline.csv")
    fieldnames = ["row_id", *class_labels]
    class_cols = class_labels
    print("mode:", BASELINE_MODE)
    print("synthetic class count:", len(class_labels))

print("output path:", output_path)
print("row count:", len(rows))
print("column count:", len(fieldnames))

In [ ]:
if BASELINE_MODE == "REAL_SAMPLE_ZERO_BASELINE":
    validate_real_sample_zero_rows(rows, fieldnames)
    written = write_submission_csv_with_fieldnames(rows, output_path, fieldnames)
else:
    validate_submission_rows(rows, class_cols)
    written = write_submission_csv(rows, output_path, class_cols)

print("wrote:", written)
print("exists:", written.exists())
print("size bytes:", written.stat().st_size if written.exists() else None)
print("first row_id:", rows[0]["row_id"])
print("last row_id:", rows[-1]["row_id"])
print("header preview:", list(fieldnames)[:8], "...")

with written.open("r", encoding="utf-8") as handle:
    for idx in range(3):
        line = handle.readline().rstrip()
        print(f"csv line {idx}:", line[:500])

_m03_elapsed = time.perf_counter() - _m03_t0
print("runtime seconds (notebook path):", round(_m03_elapsed, 3))

In [ ]:
print("=== M03 baseline claim summary ===")
print("BASELINE_MODE:", BASELINE_MODE)

if BASELINE_MODE == "SYNTHETIC_FALLBACK_ONLY":
    print("No real Kaggle submission path was proven.")
    print("Output:", output_path)
else:
    print("Generated /kaggle/working/submission.csv from real sample_submission.csv schema.")
    print("This does not prove leaderboard submission or score unless submitted and recorded.")
    print("Output:", output_path)

print("/kaggle/working/submission.csv mentioned for real sample mode when file exists on Kaggle.")